# Model Training

Trains a small CNN on generated puzzle boards to predict next moves. Saves weights to zip_cnn.pth.

## Imports & Setup

In [15]:
import math
import random
import time
from dataclasses import dataclass, field
from enum import IntEnum
from pathlib import Path
from typing import List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# Device selection
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'pytorch {torch.__version__}')
print(f'using {device}')
if device.type == 'cuda':
    print(torch.cuda.get_device_name(0).lower())


PyTorch Version : 2.13.0+cpu
Active Device   : cpu
Running on multi-threaded CPU.


## Board Data Structures

In [16]:
class CellType(IntEnum):
    EMPTY: int = 0
    WALL: int = 1
    START: int = 2
    NUMBERED: int = 3
    GOAL: int = 4

MOVES: List[Tuple[int, int]] = [
    (-1, 0),  # 0 = UP
    (1, 0),   # 1 = DOWN
    (0, -1),  # 2 = LEFT
    (0, 1),   # 3 = RIGHT
]
MOVE_NAMES = ['UP', 'DOWN', 'LEFT', 'RIGHT']

@dataclass(slots=True)
class Cell:
    row: int
    col: int
    cell_type: int
    number: Optional[int] = None
    pixel_center: Tuple[int, int] = (0, 0)

@dataclass(slots=True)
class GameState:
    grid: np.ndarray
    numbers: np.ndarray
    start: Tuple[int, int]
    goal: Tuple[int, int]
    cells: List[Cell] = field(default_factory=list)
    canvas_bbox: Tuple[int, int, int, int] = (0, 0, 0, 0)
    cell_size: int = 40

    @property
    def shape(self) -> Tuple[int, int]:
        return self.grid.shape

def move_to_action(dr: int, dc: int) -> int:
    if (dr, dc) == (-1, 0): return 0
    if (dr, dc) == (1, 0):  return 1
    if (dr, dc) == (0, -1): return 2
    if (dr, dc) == (0, 1):  return 3
    raise ValueError(f'Invalid direction: ({dr}, {dc})')


## Generate Random Boards
Creates random solvable boards using random walks to generate training data.

In [17]:
def generate_synthetic_state(
    rows: int = 6,
    cols: int = 6,
    num_waypoints: int = 3,
    seed: Optional[int] = None,
) -> Tuple[GameState, List[Tuple[int, int]]]:
    rng = random.Random(seed)
    for _ in range(300):
        visited = np.zeros((rows, cols), dtype=bool)
        sr, sc = rng.randint(0, rows - 1), rng.randint(0, cols - 1)
        path = [(sr, sc)]
        visited[sr, sc] = True
        target_len = int(rows * cols * (0.65 + rng.random() * 0.25))

        while len(path) < target_len:
            r, c = path[-1]
            neighbors = []
            for dr, dc in MOVES:
                nr, nc = r + dr, c + dc
                if 0 <= nr < rows and 0 <= nc < cols and not visited[nr, nc]:
                    neighbors.append((nr, nc))
            if not neighbors:
                break
            nr, nc = rng.choice(neighbors)
            path.append((nr, nc))
            visited[nr, nc] = True

        if len(path) < max(8, num_waypoints * 3):
            continue

        # Place sequential waypoints along the solution path
        wp_count = min(num_waypoints, len(path) // 3)
        wp_indices = [0, len(path) - 1]
        if wp_count > 2:
            interior = list(range(2, len(path) - 2, max(1, len(path) // wp_count)))
            rng.shuffle(interior)
            wp_indices.extend(interior[:wp_count - 2])
        wp_indices = sorted(set(wp_indices))

        grid = np.full((rows, cols), int(CellType.WALL), dtype=np.int32)
        cells: List[Cell] = []
        numbered_cells: List[Cell] = []
        path_set = set(path)

        for r in range(rows):
            for c in range(cols):
                if (r, c) in path_set:
                    idx = path.index((r, c))
                    if idx in wp_indices:
                        wp_num = wp_indices.index(idx) + 1
                        if wp_num == 1:
                            ct = int(CellType.START)
                        elif wp_num == len(wp_indices):
                            ct = int(CellType.GOAL)
                        else:
                            ct = int(CellType.NUMBERED)
                        grid[r, c] = ct
                        cell = Cell(row=r, col=c, cell_type=ct, number=wp_num,
                                    pixel_center=(c * 40 + 20, r * 40 + 20))
                        cells.append(cell)
                        if ct == int(CellType.NUMBERED):
                            numbered_cells.append(cell)
                    else:
                        grid[r, c] = int(CellType.EMPTY)
                        cells.append(Cell(row=r, col=c, cell_type=int(CellType.EMPTY),
                                          pixel_center=(c * 40 + 20, r * 40 + 20)))
                else:
                    cells.append(Cell(row=r, col=c, cell_type=int(CellType.WALL),
                                      pixel_center=(c * 40 + 20, r * 40 + 20)))

        start = path[0]
        goal = path[-1]
        numbers = np.array(sorted({c.number for c in numbered_cells if c.number is not None}), dtype=np.int32)
        state = GameState(
            grid=grid,
            numbers=numbers,
            start=start,
            goal=goal,
            cells=cells,
            canvas_bbox=(0, 0, cols * 40, rows * 40),
            cell_size=40,
        )
        return state, path
    raise RuntimeError('Failed to generate a solvable synthetic board within iteration budget.')

# Generate and inspect a demonstration board
demo_state, demo_path = generate_synthetic_state(rows=6, cols=6, num_waypoints=4, seed=42)
print(f'sample board {demo_state.shape[0]}x{demo_state.shape[1]} path length {len(demo_path)}')

# Visualize board
for r in range(demo_state.shape[0]):
    line = ''
    for c in range(demo_state.shape[1]):
        ct = demo_state.grid[r, c]
        if ct == int(CellType.WALL): line += ' #'
        elif ct == int(CellType.START): line += ' S'
        elif ct == int(CellType.GOAL): line += ' G'
        elif ct == int(CellType.NUMBERED):
            matching = [x.number for x in demo_state.cells if x.row == r and x.col == c and x.number]
            line += f' {matching[0]}'
        else: line += ' .'
    print(line)


Synthetic Board Generated: 6x6
Start: (5, 0) -> Goal: (0, 0) | Path Length: 18
 [G] (3)  .   .   .   . 
  .   .   .   .   .   . 
  #   .   .   #   #   # 
  #   .   #   #   #   # 
  #  (2)  #   #   #   # 
 [S]  .   #   #   #   # 


## CNN Model
A 4-layer conv net with a policy head (move direction) and a value head.

In [18]:
NUM_CHANNELS: int = 4
NUM_ACTIONS: int = 4
WEIGHTS_PATH: Path = Path('zip_cnn.pth')

class ZipCNN(nn.Module):
    def __init__(self, in_channels: int = NUM_CHANNELS, num_actions: int = NUM_ACTIONS) -> None:
        super().__init__()
        self.in_channels = in_channels
        self.num_actions = num_actions

        # Shared Convolutional Trunk
        self.conv1 = nn.Conv2d(in_channels, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.conv3 = nn.Conv2d(64, 64, 3, padding=1)
        self.conv4 = nn.Conv2d(64, 64, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.bn2 = nn.BatchNorm2d(64)
        self.bn3 = nn.BatchNorm2d(64)
        self.bn4 = nn.BatchNorm2d(64)

        # Policy Head (Direction probabilities)
        self.policy_conv = nn.Conv2d(64, 2, 1)
        self.policy_fc = nn.Linear(2, 128)
        self.policy_head = nn.Linear(128, num_actions)

        # Value Head (State value in [-1, +1])
        self.value_conv = nn.Conv2d(64, 1, 1)
        self.value_fc = nn.Linear(1, 128)
        self.value_head = nn.Linear(128, 1)

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        h = F.relu(self.bn1(self.conv1(x)))
        h = F.relu(self.bn2(self.conv2(h)))
        h2 = F.relu(self.bn3(self.conv3(h)))
        h = F.relu(self.bn4(self.conv4(h2))) + h2  # Residual bypass

        # Policy path
        p_map = self.policy_conv(h)
        p_flat = p_map.mean(dim=(2, 3))
        p = F.relu(self.policy_fc(p_flat))
        policy_logits = self.policy_head(p)

        # Value path
        v_map = self.value_conv(h)
        v_flat = v_map.mean(dim=(2, 3))
        v = F.relu(self.value_fc(v_flat))
        value = torch.tanh(self.value_head(v))

        return policy_logits, value

    @torch.no_grad()
    def predict(self, state_tensor: torch.Tensor) -> Tuple[torch.Tensor, float]:
        if state_tensor.dim() == 3:
            state_tensor = state_tensor.unsqueeze(0)
        self.eval()
        logits, value = self.forward(state_tensor)
        policy_probs = F.softmax(logits, dim=1).squeeze(0)
        return policy_probs, float(value.item())

net = ZipCNN()
param_count = sum(p.numel() for p in net.parameters())
print(f'model ready with {param_count} parameters')


ZipCNN Architecture initialized successfully.
Total Trainable Parameters: 95,464


## Prepare Training Data
Converts solved boards into state tensors and target moves.

In [19]:
def build_input_tensor(state: GameState, cur_r: int, cur_c: int, visited: np.ndarray, next_wp: Optional[Tuple[int, int]]) -> torch.Tensor:
    rows, cols = state.shape
    tensor = torch.zeros(NUM_CHANNELS, rows, cols, dtype=torch.float32)
    # Channel 0: Walls
    tensor[0] = torch.from_numpy((state.grid == int(CellType.WALL)).astype(np.float32))
    # Channel 1: Current position
    tensor[1, cur_r, cur_c] = 1.0
    # Channel 2: Visited cells
    tensor[2] = torch.from_numpy(visited.astype(np.float32))
    # Channel 3: Next waypoint target
    if next_wp is not None:
        tensor[3, next_wp[0], next_wp[1]] = 1.0
    return tensor

def generate_training_samples(num_games: int = 150, board_sizes: List[int] = [5, 6, 7, 8]) -> list:
    samples = []
    print(f'generating {num_games} boards')
    t0 = time.perf_counter()

    for i in range(num_games):
        size = random.choice(board_sizes)
        n_wp = random.choice([2, 3, 4, 5])
        try:
            state, true_path = generate_synthetic_state(rows=size, cols=size, num_waypoints=n_wp, seed=i)
        except Exception:
            continue

        # Extract sequentially ordered waypoints
        waypoints = [c for c in state.cells if c.cell_type in [int(CellType.START), int(CellType.NUMBERED), int(CellType.GOAL)]]
        waypoints.sort(key=lambda x: x.number or 0)
        wp_coords = [(w.row, w.col) for w in waypoints]

        visited = np.zeros((size, size), dtype=bool)
        wp_idx = 0

        for step in range(len(true_path) - 1):
            r, c = true_path[step]
            visited[r, c] = True
            if wp_idx < len(wp_coords) and (r, c) == wp_coords[wp_idx]:
                wp_idx += 1

            next_target = wp_coords[wp_idx] if wp_idx < len(wp_coords) else state.goal
            state_tensor = build_input_tensor(state, r, c, visited, next_target)

            nr, nc = true_path[step + 1]
            dr, dc = nr - r, nc - c
            try:
                action = move_to_action(dr, dc)
            except ValueError:
                continue

            policy_target = torch.zeros(NUM_ACTIONS, dtype=torch.float32)
            policy_target[action] = 1.0

            # Progressive value scaling: closer to goal = higher reward
            progress = step / max(1, len(true_path) - 1)
            value_target = -1.0 + 2.0 * progress

            samples.append((state_tensor, policy_target, value_target))

    elapsed = time.perf_counter() - t0
    print(f'created {len(samples)} samples in {elapsed:.1f}s')
    return samples

# Build samples
samples = generate_training_samples(num_games=100)
print(f'sample shape {list(samples[0][0].shape)}')


Generating self-play dataset from 100 synthetic games...
Done! Generated 2,128 training samples in 0.06s.
Sample state shape : torch.Size([4, 8, 8])
Sample policy target: tensor([0., 0., 0., 1.])
Sample value target : -1.00


## Training Loop
Trains the model on the generated boards.

In [20]:
def pad_tensor(t: torch.Tensor, target_h: int, target_w: int) -> torch.Tensor:
    c, h, w = t.shape
    if h == target_h and w == target_w:
        return t
    padded = torch.zeros(c, target_h, target_w, dtype=t.dtype)
    padded[:, :h, :w] = t
    return padded

def train_model(
    model: ZipCNN,
    samples: list,
    epochs: int = 15,
    batch_size: int = 64,
    lr: float = 1e-3,
    device: torch.device = device,
) -> ZipCNN:
    model = model.to(device)
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)

    n = len(samples)
    print(f'training for {epochs} epochs on {device}')

    for epoch in range(1, epochs + 1):
        perm = torch.randperm(n)
        total_pl = 0.0
        total_vl = 0.0
        batches = 0

        for i in range(0, n, batch_size):
            batch_indices = perm[i:i + batch_size]
            sub_samples = [samples[j] for j in batch_indices.tolist()]

            bh = max(s[0].shape[1] for s in sub_samples)
            bw = max(s[0].shape[2] for s in sub_samples)

            s_b = torch.stack([pad_tensor(s[0], bh, bw) for s in sub_samples]).to(device)
            p_b = torch.stack([s[1] for s in sub_samples]).to(device)
            v_b = torch.tensor([s[2] for s in sub_samples], dtype=torch.float32, device=device).unsqueeze(1)

            optimizer.zero_grad()
            logits, v = model(s_b)

            # Composite Loss
            loss_policy = -(p_b * F.log_softmax(logits, dim=1)).sum(dim=1).mean()
            loss_value = F.mse_loss(v, v_b)
            loss = loss_policy + loss_value

            loss.backward()
            optimizer.step()

            total_pl += loss_policy.item()
            total_vl += loss_value.item()
            batches += 1

        avg_pl = total_pl / max(1, batches)
        avg_vl = total_vl / max(1, batches)
        print(f'epoch {epoch} of {epochs} loss {avg_pl + avg_vl:.4f}')

    model.eval()
    return model

# Train the network
trained_net = train_model(net, samples, epochs=15, batch_size=64, lr=1e-3)


Starting training on cpu for 15 epochs (Batch size: 64)...

Epoch [ 1/15] -> Policy Loss: 1.3611 | Value Loss: 0.0968 | Total: 1.4579
Epoch [ 2/15] -> Policy Loss: 1.2446 | Value Loss: 0.0330 | Total: 1.2776
Epoch [ 3/15] -> Policy Loss: 1.0019 | Value Loss: 0.0301 | Total: 1.0319
Epoch [ 4/15] -> Policy Loss: 0.7004 | Value Loss: 0.0288 | Total: 0.7293
Epoch [ 5/15] -> Policy Loss: 0.4903 | Value Loss: 0.0262 | Total: 0.5165
Epoch [ 6/15] -> Policy Loss: 0.3882 | Value Loss: 0.0250 | Total: 0.4131
Epoch [ 7/15] -> Policy Loss: 0.3122 | Value Loss: 0.0227 | Total: 0.3349
Epoch [ 8/15] -> Policy Loss: 0.2635 | Value Loss: 0.0210 | Total: 0.2845
Epoch [ 9/15] -> Policy Loss: 0.2269 | Value Loss: 0.0281 | Total: 0.2550
Epoch [10/15] -> Policy Loss: 0.1955 | Value Loss: 0.0244 | Total: 0.2199
Epoch [11/15] -> Policy Loss: 0.1717 | Value Loss: 0.0189 | Total: 0.1906
Epoch [12/15] -> Policy Loss: 0.1512 | Value Loss: 0.0220 | Total: 0.1731
Epoch [13/15] -> Policy Loss: 0.1559 | Value Loss: 0

## Test & Save Weights
Runs a sample prediction and saves weights to zip_cnn.pth.

In [21]:
# Run test inference on the demo state
test_tensor = samples[0][0].to(device)
probs, val = trained_net.predict(test_tensor)

print(f'test value {val:.3f}')
for name, prob in zip(MOVE_NAMES, probs.tolist()):
    print(f'{name.lower()} {prob:.2f}')

# Export weights
torch.save(trained_net.state_dict(), WEIGHTS_PATH)
print(f'saved weights to {WEIGHTS_PATH.name}')


=== Inference Test on Sample State ===
Predicted Value: -0.9261
Action Probabilities:
  UP    :   2.2%  [                              ]
  DOWN  :  21.3%  [======                        ]
  LEFT  :  75.4%  [======================        ]
  RIGHT :   1.1%  [                              ]

Model successfully saved to C:\Users\shiva\Documents\Zip\linkedin_zip_solver\zip_cnn.pth
Weight File Size: 386.3 KB
